#Cross Attention

In [1]:
import torch
import math
import torch.nn as nn

In [2]:
class CrossAttention(nn.Module):
  def __init__(self,d_model):
    super().__init__()

    self.Wq=nn.Linear(d_model,d_model)
    self.Wk=nn.Linear(d_model,d_model)
    self.Wv=nn.Linear(d_model,d_model)

  def forward(self,decoder,encoder):
    Q=self.Wq(decoder)
    K=self.Wk(encoder)
    V=self.Wv(encoder)

    scores=torch.matmul(
        Q,K.transpose(-2,-1)
    )

    scores=scores/math.sqrt(K.size(-1))

    weights=torch.softmax(
        scores,dim=-1
    )
    output=torch.matmul(
        weights,V
    )

    return output

In [3]:
d_model = 8

model = CrossAttention(d_model)

# batch_size = 2
# decoder_seq_len = 3
# encoder_seq_len = 5

decoder = torch.randn(2, 3, 8)

encoder = torch.randn(2, 5, 8)

output = model(decoder, encoder)

print("=" * 50)
print("Decoder Shape :", decoder.shape)
print("Encoder Shape :", encoder.shape)
print("Output Shape  :", output.shape)
print("=" * 50)

print(output)

Decoder Shape : torch.Size([2, 3, 8])
Encoder Shape : torch.Size([2, 5, 8])
Output Shape  : torch.Size([2, 3, 8])
tensor([[[-0.0028,  0.6158,  0.0294, -0.5087,  0.1560,  0.3494,  0.5931,
           0.1250],
         [ 0.0722,  0.5132, -0.0708, -0.4547,  0.0293,  0.3573,  0.6335,
           0.0401],
         [ 0.0390,  0.5390,  0.0646, -0.4685,  0.1254,  0.3449,  0.5851,
           0.0588]],

        [[-0.1902,  0.6561,  0.3467, -0.7665,  0.7688,  0.6383,  1.0464,
           0.2334],
         [-0.2050,  0.5417,  0.5751, -0.7193,  0.9591,  0.6033,  0.8848,
           0.0441],
         [-0.1809,  0.5927,  0.4254, -0.7320,  0.7956,  0.6177,  0.9759,
           0.1472]]], grad_fn=<UnsafeViewBackward0>)


#Multi-Head Cross Attention

In [10]:
class MultiHeadCrossAttention(nn.Module):
  def __init__(self,d_model,num_heads):
    super().__init__()
    self.d_model=d_model
    self.num_heads=num_heads
    self.head_dim=d_model//num_heads

    self.Wq=nn.Linear(d_model,d_model)
    self.Wk=nn.Linear(d_model,d_model)
    self.Wv=nn.Linear(d_model,d_model)

    self.fc=nn.Linear(d_model,d_model)

  def forward(self,decoder_output,encoder_output):

    batch_size=decoder_output.size(0)

    decoder_seq_len=decoder_output.size(1)
    encoder_seq_len=encoder_output.size(1)

    #Linear Projection
    Q=self.Wq(decoder_output)
    K=self.Wk(encoder_output)
    V=self.Wv(encoder_output)

    #Split into heads
    Q=Q.view(
        batch_size,
        decoder_seq_len,
        self.num_heads,
        self.head_dim
    ).transpose(1,2)
    K=K.view(
        batch_size,
        encoder_seq_len,
        self.num_heads,
        self.head_dim
    ).transpose(1,2)
    V=V.view(
        batch_size,
        encoder_seq_len,
        self.num_heads,
        self.head_dim
    ).transpose(1,2)

    #attention socre
    scores=torch.matmul(
        Q,
        K.transpose(-2,-1)
    )
    #scaling
    scores=scores/math.sqrt(self.head_dim)

    #attention weight(softmax)
    weights=torch.softmax(
        scores,dim=-1
    )
    #context(output)
    output=torch.matmul(
        weights,V
    )

    # merge Heads
    output=output.transpose(
        1,2
    ).contiguous().view(
        batch_size,decoder_seq_len,self.d_model
    )

    output=self.fc(output)
    return output


In [11]:
batch_size=2
decoder_seq_len=5
encoder_seq_len=3

d_model=16
num_heads=4

encoder=torch.randn(
    batch_size,encoder_seq_len,d_model
)

decoder=torch.randn(
    batch_size,decoder_seq_len,d_model
)
crossAttention=MultiHeadCrossAttention(
    d_model,num_heads
)
output=crossAttention(decoder,encoder)

print("Encoder output Shape: ",encoder.shape)
print("decoder output Shape: ",decoder.shape)
print("Final output Shape: ",output.shape)

Encoder output Shape:  torch.Size([2, 3, 16])
decoder output Shape:  torch.Size([2, 5, 16])
Final output Shape:  torch.Size([2, 5, 16])




```
Encoder Output
(2,5,16)
        │
        │
      K,V
        │
        ▼
----------------------------
Multi-Head Cross Attention
----------------------------
        ▲
        │
        Q
Decoder Output
(2,3,16)

After Spliting Heads

Q : (2,4,3,4)

K : (2,4,5,4)

V : (2,4,5,4)

Attention Scores
Q × Kᵀ

(2,4,3,4)

×

(2,4,4,5)

↓

(2,4,3,5)

softmax -->(2,4,3,5)

weighted sum

(2,4,3,5)

×

(2,4,5,4)

↓

(2,4,3,4)

Merge Heads
(2,4,3,4)

↓

(2,3,16)

```



In [8]:
8/4

2.0

In [9]:
8//4

2